In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/q17_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# 1. Filter PART to get the relevant keys on GPU
target_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

# 2. Early‐filter LINEITEM to just those keys
li = lineitem[lineitem.L_PARTKEY.isin(target_keys)]

# 3. Compute the 20%‐scaled average quantity per key using transform (avoids a merge)
li['avg'] = li.groupby('L_PARTKEY').L_QUANTITY.transform('mean') * 0.2

# 4. Apply the quantity filter and compute the final metric entirely on GPU
good = li[li.L_QUANTITY < li.avg]
total = pd.DataFrame({'avg_yearly': [good.L_EXTENDEDPRICE.sum() / 7.0]})